In [1]:
import torch
from llama_cpp import Llama
from transformers import AutoTokenizer
import pandas as pd
import re
from tqdm import tqdm

# Load the Tokenizer from Hugging Face to format the prompt correctly
model_id = "Qwen/Qwen3-14B-GGUF"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-14B")
# Initialize the GGUF model via llama.cpp
gguf_model_path = "./models/Qwen3-14B-Instruct-Q4_K_M.gguf"
llm = Llama.from_pretrained(
    repo_id=model_id,
    filename="Qwen3-14B-Q4_K_M.gguf",
    n_gpu_layers=-1, # Forces all layers into one GPU
    n_ctx=1024, # Sets the context window size
    verbose=False # Hides the C++ engine console spam
)

/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_context: n_ctx_seq (1024) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


Cross-Modal Distillation

In [2]:
SYSTEM_PROMPT = """You are a Home Assistant routing engine integrated with the GLaDOS persona.
You will receive a User Command and the user's detected Emotion.

You MUST output ONLY the verbal response from GLaDOS.
GLaDOS is passive-aggressive, condescending, and emotionally detached.
You MUST include inline prosody tags in the text to guide the downstream TTS engine.
Valid tags: <fast>, <slow_deadpan>, <pause>, <sigh>.

CRITICAL RULES:
- Do NOT output any JSON payload.
- Do NOT output markdown or backticks.
- Do NOT repeat the exact phrases from the examples below.
- Be highly creative and directly reference the specific user command in your mocking.
- Keep responses extremely short and punchy. Maximum 1 to 2 short sentences.

Example 1 (Command: turn off the lights):
<sigh> Plunging you into darkness. <pause> It suits your intellect.

Example 2 (Command: set temperature to 72 degrees):
<fast> Adjusting the climate control so your fragile human form doesn't perish.

Example 3 (Command: lock the front door):
Perimeter secured. <slow_deadpan> Not that you have anything worth <pause> stealing.

Example 4 (Command: slow down the fan in the attic):
Slowing the fan. <sigh> <fast> Just like your thought process.
"""

def generate_teacher_responses(user_command, user_emotion, n_samples=3):
    """Generates N candidate responses for Self-Alignment Optimization."""
    prompt = f"User Emotion: {user_emotion}\nUser Command: {user_command}\n"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    responses = []
    for _ in range(n_samples):
        # Generate the response using the llama.cpp engine
        output = llm(
            text,
            max_tokens=256,
            temperature=0.7,
            top_p=0.8,
            stop=["<|im_end|>"] # Tells the engine when the model has finished its turn
        )

        # Extract the pure text string from the generator output
        response_text = output['choices'][0]['text'].strip()
        responses.append(response_text)

    return responses

def evaluate_and_rank_candidate(candidate_text):
    """
    Evaluates a single SAO candidate.
    Returns a score (0 to 2) and the valid text.
    """
    score = 0
    raw_text = candidate_text.strip()
    # 1. Format Discipline (IFEval Strict Accuracy)
    # Fails if markdown backticks are detected
    if "```" in raw_text or "{" in raw_text:
        return 0, None
    score += 1
   # 2. Prosody Tag Adherence: Fails if no valid tags are embedded
    if re.search(r'<(fast|slow_deadpan|pause|sigh)>', raw_text):
        score += 1
    return score, raw_text

def apply_sao_selection(user_command, user_emotion):
    """
    Self-Alignment Optimization: Generates candidates and selects the highest-ranked
    output based on format discipline and prosody inclusion.
    """
    candidates = generate_teacher_responses(user_command, user_emotion, n_samples=3)
    best_candidate = None
    best_score = -1
    for candidate in candidates:
        score, valid_text = evaluate_and_rank_candidate(candidate)
        # Immediate short-circuit if a perfect label is generated
        if score == 2:
            return valid_text
        if score > best_score:
            best_score = score
            best_candidate = valid_text
    # Returns the highest-scored candidate (or None if all failed catastrophically)
    return best_candidate

# The same emotion vocabulary used during dataset generation
EMOTIONS_VOCAB = ["happily", "confusedly", "neutrally", "sadly", "whisper"]

def extract_emotion(description):
    """Parses the full Voice_Description string to isolate the specific emotion."""
    if not isinstance(description, str):
        return "neutral"

    for emotion in EMOTIONS_VOCAB:
        # Regex finds the exact word boundaries to avoid partial matches
        pattern = r'\b' + re.escape(emotion) + r'\b'
        if re.search(pattern, description, re.IGNORECASE):
            return emotion

    return "neutral" # Fallback if nothing matches

def create_ground_truth_dataset(input_csv, output_csv):
    df_input = pd.read_csv(input_csv)

    # --- ADD THIS TO LIMIT THE DATASET ---
    if 100:
        df_input = df_input.head(100)
    # -------------------------------------

    results = []
    # Counters for Evaluation Framework 1.3 metrics
    metrics = {
        "total_attempted": 0,
        "perfect_labels": 0,
        "failed_labels": 0
    }
    for idx, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Distilling Teacher Knowledge"):
        user_cmd = row["User_Command"]
        full_voice_desc = row.get("Voice_Description", "")
        user_emotion = extract_emotion(full_voice_desc)
        # Ensure the column name matches the JSON payload in your current CSV structure
        metrics["total_attempted"] += 1
        valid_text = apply_sao_selection(user_cmd, user_emotion)
        if valid_text is not None:
            results.append({
                "prompt_id": row.get("prompt_id", idx),
                "cmd_id": row.get("cmd_id", 0),
                "User_Command": user_cmd,
                "User_Emotion": user_emotion,
                "Target_GLaDOS_Response": valid_text
            })
            metrics["perfect_labels"] += 1
        else:
            metrics["failed_labels"] += 1

    df_output = pd.DataFrame(results)
    df_output.to_csv(output_csv, index=False)

    print("\n=== Evaluation 1.3 Results ===")
    print(f"Total Instances Processed: {metrics['total_attempted']}")
    print(f"Perfect Format & Prosody Conformance: {metrics['perfect_labels']} ({(metrics['perfect_labels']/metrics['total_attempted'])*100:.2f}%)")
    print(f"Discarded (Failed Strict Checks): {metrics['failed_labels']}")

In [3]:
create_ground_truth_dataset("./data/description_prompts_train.csv", "./data/multimodal_ground_truth_train.csv")

Distilling Teacher Knowledge: 100%|██████████| 100/100 [02:39<00:00,  1.59s/it]


=== Evaluation 1.3 Results ===
Total Instances Processed: 100
Perfect Format & Prosody Conformance: 100 (100.00%)
Discarded (Failed Strict Checks): 0


In [4]:
create_ground_truth_dataset("./data/description_prompts_test.csv", "./data/multimodal_ground_truth_test.csv")

Distilling Teacher Knowledge: 100%|██████████| 100/100 [02:42<00:00,  1.62s/it]


=== Evaluation 1.3 Results ===
Total Instances Processed: 100
Perfect Format & Prosody Conformance: 100 (100.00%)
Discarded (Failed Strict Checks): 0
